# Run Model

Single-country simulation runner with standard macro, benchmark, permanent-income, sector, and firm-credit diagnostics. Configure the inputs, run the workflow once, then inspect the manifest and diagnostic sections below.

## Companion notebooks

- `run_model_exploration.ipynb` — exploratory firm, household, reader, and ratio analysis
- `run_sensitivity.ipynb` — parameter sensitivity
- `run_mpc.ipynb` — household MPC experiment
- `run_irf.ipynb` — macro impulse responses
- `run_model_legacy_2026-08-18.ipynb` — preserved historical notebook; do not use for new work

In [ ]:
# Setup
%load_ext autoreload
%autoreload 2

import numpy as np

from src.monte_carlo import run_seeded_monte_carlo
from config import (
    BALANCE_SHEET_COLUMNS,
    FIGURE_SIZES,
    FISCAL_COLUMNS,
    LABOUR_COLUMNS,
    MACRO_COLUMNS,
    POLICY_COLUMNS,
    SCENARIO_PRESETS,
)
from src.notebook_state import run_notebook_workflow, validate_notebook_state
from src.notebook_workflow import (
    NotebookRunConfig,
    build_permanent_income_forecast_contribution_table,
    build_permanent_income_log_ratio_decomposition_df,
    plot_permanent_income_log_ratio_decomposition,
)
from src.visual_helpers import (
    firm_sector_groups_table,
    plot_agent_timeseries,
    plot_cumulative_insolvent_firms_by_sector,
    plot_firm_credit_to_equity_and_capital,
    plot_mc,
    plot_output,
    plot_sector_tfp_investment_desired_mb_mc_ratio,
)

## Inputs

All execution switches are centralized here. Sensitivity, MPC, and IRF switches are in their dedicated notebooks.

In [ ]:
RUN_BENCHMARK = True
RUN_MONTE_CARLO = False
# SCENARIO_NAME = "calibrated_consumption"
SCENARIO_NAME = "change_sectoral_weights"

run_config = NotebookRunConfig(
    seed=168,
    t_max=50,
    country_iso3="FRA",
    run_benchmark=RUN_BENCHMARK,
    force_rebuild_data=True,
    force_rerun_benchmark=True,
    benchmark_overrides=None,
)
scenario_overrides = SCENARIO_PRESETS[SCENARIO_NAME]




## Run simulation

In [ ]:
# In the first run, state is none. After, run_notebook_workflow uses prepared from the previous state, preventing reloading it.
state = globals().get("state")

state = run_notebook_workflow(
    run_config,
    scenario_name=SCENARIO_NAME,
    scenario_overrides=scenario_overrides,
    previous_state=state,
)

# Familiar aliases; `state` remains the authoritative carrier.
COUNTRY = state.country_code
prepared = state.prepared
data = prepared.data
cfg = prepared.cfg
simulation = state.simulation
model = state.model
df_scenario = state.df_scenario
benchmark = state.benchmark
df_benchmark = state.df_benchmark

validate_notebook_state(state)

In [ ]:
{
    "country": COUNTRY,
    "seed": cfg.seed,
    "t_max": cfg.t_max,
    "scenario": state.scenario_name,
    "model_h5": str(simulation.model_h5_path),
    "benchmark": benchmark is not None,
    "manifest": str(state.manifest_path),
}


## Macro, fiscal, policy, labour, and balance sheets

In [ ]:
plot_output(
    df=df_scenario[list(MACRO_COLUMNS)],
    no_rows=5,
    no_cols=4,
    country_code=COUNTRY,
    line_color="#1f77b4",
    **FIGURE_SIZES["benchmark"],
)
plot_output(df=df_scenario[list(FISCAL_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(POLICY_COLUMNS)], no_rows=4, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(LABOUR_COLUMNS)], no_rows=2, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
plot_output(df=df_scenario[list(BALANCE_SHEET_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


## Benchmark comparison

In [ ]:
if df_benchmark is None:
    print("Benchmark disabled in Inputs.")
else:
    plot_output(df=df_scenario[list(MACRO_COLUMNS)], df_compare=df_benchmark[list(MACRO_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_scenario[list(FISCAL_COLUMNS)], df_compare=df_benchmark[list(FISCAL_COLUMNS)], no_rows=5, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_scenario[list(POLICY_COLUMNS)], df_compare=df_benchmark[list(POLICY_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])
    plot_output(df=df_scenario[list(BALANCE_SHEET_COLUMNS)], df_compare=df_benchmark[list(BALANCE_SHEET_COLUMNS)], no_rows=3, no_cols=4, country_code=COUNTRY, **FIGURE_SIZES["benchmark"])


## Permanent-income decomposition

In [ ]:
decomposition = build_permanent_income_log_ratio_decomposition_df(
    simulation,
    country_code=COUNTRY,
    reducer="mean",
    include_log_real_pc_income=True,
)
plot_permanent_income_log_ratio_decomposition(
    simulation,
    country_code=COUNTRY,
    columns=["ln_y_p_over_y", "common_log_ratio"],
    reducer="mean",
)

fig = plot_permanent_income_log_ratio_decomposition(
    simulation,
    country_code=COUNTRY,
    columns=["real_pc_income_idx"],
    reducer="mean",
    title='real_pc_income_idx',
    show=True,
)


contributions = build_permanent_income_forecast_contribution_table(
    simulation,
    country_code=COUNTRY,
    periods=list(range(9)),
    include_fixed=False,
)
contributions


## Sector and firm-credit diagnostics

In [ ]:
firm_sector_groups_table(model, COUNTRY)
plot_cumulative_insolvent_firms_by_sector(df_scenario)

credit_panels = [
    ["total_target_short_term_credit", "total_received_short_term_credit"],
    ["total_target_long_term_credit", "total_received_long_term_credit"],
    "short_term_loan_debt",
    "long_term_loan_debt",
]
plot_agent_timeseries(
    model,
    COUNTRY,
    "firms",
    variables=credit_panels,
    agg="sum",
    no_cols=2,
    show_legend=False,
    **FIGURE_SIZES["dense"],
)
plot_firm_credit_to_equity_and_capital(model, COUNTRY, show=True, return_df=False)
plot_sector_tfp_investment_desired_mb_mc_ratio(model, COUNTRY)


## Optional Monte Carlo

In [ ]:
if RUN_MONTE_CARLO:
    rng = np.random.default_rng(run_config.seed)
    mc_seeds = rng.choice(np.arange(1000), size=50, replace=False).tolist()
    mc = run_seeded_monte_carlo(
        datawrapper=data,
        country_configurations=state.country_configurations,
        country_code=COUNTRY,
        seeds=mc_seeds,
        t_max=cfg.t_max,
        n_jobs=-1,
        backend="loky",
        batch_size=1,
    )
    plot_mc(mc=mc, cols=list(MACRO_COLUMNS), no_cols=4, country_code=COUNTRY)
else:
    print("Set RUN_MONTE_CARLO = True to run seeded Monte Carlo simulations.")
